# V10 publication-figure generator for MBEC

Run this notebook **after** V10 has completed. It reads the saved V10 CSV files;
it does not retrain any classifier and does not contain manually entered result
values.

Outputs follow the current *Medical & Biological Engineering & Computing*
artwork guidance:

- main figures at 174 mm width;
- Arial/Helvetica-style sans-serif lettering, 8--12 pt at final size;
- lowercase panel labels;
- EPS vector output with embedded TrueType fonts;
- RGB TIFF combination artwork at 600 dpi;
- PNG previews for convenient inspection;
- graphical abstract at 32.93 x 37.63 mm;
- captions kept outside the artwork.

Generated items:

1. Fig1: study framework
2. Fig2: temporal labels and causal alarm mechanism
3. Fig3: epoch-level BAcc/MCC heatmaps
4. Fig4: primary seizure-sensitivity--FAR trade-off
5. Fig5: paired patient-level F1 versus F7 MCC
6. Graphical Abstract
7. Supplementary Figs S1--S4
8. Figure manifest and caption text files


In [ ]:
# ============================================================================
# 0. CONFIGURATION, INPUT DISCOVERY AND MBEC ARTWORK STYLE
# ============================================================================
import json
import math
import re
import textwrap
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Rectangle
import numpy as np
import pandas as pd
import seaborn as sns

RESULTS_ROOT_OVERRIDE = None
SEARCH_ROOTS = [Path('/kaggle/working'), Path('/kaggle/input')]
OUTPUT_ROOT = Path('/kaggle/working/MBEC_V10_PUBLICATION_FIGURES')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

FEATURE_ORDER = [
    'F1_Handcrafted', 'F2_Graph_PLI', 'F3_Graph_wPLI',
    'F4_Graph_PLI_wPLI', 'F5_Handcrafted_PLI',
    'F6_Handcrafted_wPLI', 'F7_All',
]
FEATURE_SHORT = {
    'F1_Handcrafted': 'F1 HC',
    'F2_Graph_PLI': 'F2 PLI',
    'F3_Graph_wPLI': 'F3 wPLI',
    'F4_Graph_PLI_wPLI': 'F4 PLI+wPLI',
    'F5_Handcrafted_PLI': 'F5 HC+PLI',
    'F6_Handcrafted_wPLI': 'F6 HC+wPLI',
    'F7_All': 'F7 All',
}
MODEL_ORDER = ['RandomForest', 'XGBoost', 'SVM_RBF']
MODEL_SHORT = {'RandomForest': 'RF', 'XGBoost': 'XGB', 'SVM_RBF': 'SVM-RBF'}

# Okabe-Ito-inspired, color-blind-safe palette. Shapes and text labels preserve
# interpretability when printed in grayscale.
FEATURE_COLORS = {
    'F1_Handcrafted': '#0072B2',
    'F2_Graph_PLI': '#E69F00',
    'F3_Graph_wPLI': '#009E73',
    'F4_Graph_PLI_wPLI': '#CC79A7',
    'F5_Handcrafted_PLI': '#D55E00',
    'F6_Handcrafted_wPLI': '#56B4E9',
    'F7_All': '#6A3D9A',
}
MODEL_COLORS = {'RandomForest': '#1F77B4', 'XGBoost': '#D55E00', 'SVM_RBF': '#009E73'}
FEATURE_MARKERS = {
    'F1_Handcrafted': 'o', 'F2_Graph_PLI': 's', 'F3_Graph_wPLI': '^',
    'F4_Graph_PLI_wPLI': 'D', 'F5_Handcrafted_PLI': 'P',
    'F6_Handcrafted_wPLI': 'X', 'F7_All': '*',
}


def discover_results_root():
    if RESULTS_ROOT_OVERRIDE is not None:
        candidates = [Path(RESULTS_ROOT_OVERRIDE)]
    else:
        candidates = []
        for root in SEARCH_ROOTS:
            if root.exists():
                candidates.extend(p.parent for p in root.rglob('cohort_pooled_summary.csv'))
    required = {
        'cohort_pooled_summary.csv', 'subject_metrics.csv',
        'alarm_sensitivity_pooled_summary.csv', 'selected_features.csv',
        'tuning_summary.csv', 'inference_timing_summary.csv',
    }
    valid = sorted({p.resolve() for p in candidates
                    if required.issubset({f.name for f in p.iterdir()})})
    preferred = [p for p in valid if 'V10' in p.name.upper() or 'MULTIMODEL' in p.name.upper()]
    valid = preferred or valid
    if not valid:
        raise FileNotFoundError(
            'Could not locate a complete V10 result directory. Set RESULTS_ROOT_OVERRIDE.')
    if len(valid) > 1:
        raise RuntimeError(f'Multiple V10 result directories found: {valid}; set RESULTS_ROOT_OVERRIDE')
    return valid[0]


RESULTS_ROOT = discover_results_root()
pooled = pd.read_csv(RESULTS_ROOT / 'cohort_pooled_summary.csv')
subjects_df = pd.read_csv(RESULTS_ROOT / 'subject_metrics.csv')
alarm_pooled = pd.read_csv(RESULTS_ROOT / 'alarm_sensitivity_pooled_summary.csv')
selected_features = pd.read_csv(RESULTS_ROOT / 'selected_features.csv')
tuning = pd.read_csv(RESULTS_ROOT / 'tuning_summary.csv')
timing = pd.read_csv(RESULTS_ROOT / 'inference_timing_summary.csv')

required_models = set(MODEL_ORDER)
required_features = set(FEATURE_ORDER)
if set(pooled['Model']) != required_models:
    raise AssertionError(f'Expected models {required_models}, got {set(pooled["Model"])}')
if set(pooled['FeatureSet']) != required_features:
    raise AssertionError('V10 feature sets are incomplete')
if pooled.duplicated(['Model', 'FeatureSet']).any():
    raise AssertionError('Duplicate pooled model-feature rows')


def choose_font():
    for name in ['Arial', 'Helvetica', 'Liberation Sans', 'DejaVu Sans']:
        try:
            font_manager.findfont(name, fallback_to_default=False)
            return name
        except ValueError:
            pass
    return 'DejaVu Sans'


FONT = choose_font()
mpl.rcParams.update({
    'font.family': FONT, 'font.size': 9,
    'axes.titlesize': 9, 'axes.labelsize': 9,
    'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'legend.fontsize': 7.5, 'figure.dpi': 150,
    'savefig.facecolor': 'white', 'axes.facecolor': 'white',
    'ps.fonttype': 42, 'pdf.fonttype': 42,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': 0.8, 'xtick.major.width': 0.8, 'ytick.major.width': 0.8,
    'grid.color': '#D9DEE3', 'grid.linewidth': 0.55,
    'grid.alpha': 1.0, 'legend.frameon': False,
})
sns.set_theme(style='ticks', rc={'font.family': FONT})

MM_TO_IN = 1 / 25.4
FULL_WIDTH = 174 * MM_TO_IN
SINGLE_WIDTH = 84 * MM_TO_IN
GA_WIDTH = 32.93 * MM_TO_IN
GA_HEIGHT = 37.63 * MM_TO_IN
manifest_rows = []


def save_mbec_figure(fig, stem, width_mm, height_mm, caption_key, graphical_abstract=False):
    fig.set_size_inches(width_mm * MM_TO_IN, height_mm * MM_TO_IN)
    outputs = []
    for ext, kwargs in [
        ('eps', {}),
        ('tiff', {'dpi': 600, 'pil_kwargs': {'compression': 'tiff_lzw'}}),
        ('png', {'dpi': 300}),
    ]:
        path = OUTPUT_ROOT / f'{stem}.{ext}'
        # Do not use bbox_inches='tight': it changes the requested physical
        # dimensions and is particularly unsuitable for the graphical abstract.
        fig.savefig(path, format=ext, facecolor='white', **kwargs)
        outputs.append(str(path))
    manifest_rows.append({
        'Figure': stem, 'CaptionKey': caption_key,
        'Width_mm': width_mm, 'Height_mm': height_mm,
        'GraphicalAbstract': int(graphical_abstract),
        'Files': ' | '.join(outputs),
    })


def panel_label(ax, label):
    ax.text(0.00, 1.035, f'({label})', transform=ax.transAxes,
            fontsize=10, fontweight='bold', va='bottom', ha='left')


def rounded_box(ax, xy, width, height, text, facecolor, edgecolor='#37474F',
                fontsize=8, weight='normal', radius=0.02):
    box = FancyBboxPatch(
        xy, width, height,
        boxstyle=f'round,pad=0.008,rounding_size={radius}',
        facecolor=facecolor, edgecolor=edgecolor, linewidth=1.0)
    ax.add_patch(box)
    ax.text(xy[0] + width / 2, xy[1] + height / 2, text,
            ha='center', va='center', fontsize=fontsize, fontweight=weight)
    return box


def arrow(ax, start, end, color='#455A64'):
    ax.add_patch(FancyArrowPatch(start, end, arrowstyle='-|>', mutation_scale=11,
                                linewidth=1.1, color=color))


N_SUBJECTS = int(pooled['Subjects'].max())
N_SEIZURES = int(pooled['Seizures'].max())
print('V10 results:', RESULTS_ROOT)
print(f'Cohort: {N_SUBJECTS} subjects, {N_SEIZURES} seizures')
print('Artwork font:', FONT)


In [ ]:
# ============================================================================
# 1. FIGURE 1 — OVERALL STUDY FRAMEWORK
# ============================================================================
fig, ax = plt.subplots()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

# Journal-style cards use white surfaces and a single accent per stage.
NAVY, MUTED, BORDER = '#173B57', '#5B6975', '#C7D0D8'
ACCENTS = ['#2F80ED', '#1B998B', '#F2994A', '#7B61A8']
xs = [0.010, 0.190, 0.415, 0.660]
ws = [0.160, 0.205, 0.225, 0.330]
y0, h = 0.13, 0.74


def stage_card(x, w, number, title, accent):
    ax.add_patch(FancyBboxPatch(
        (x, y0), w, h, boxstyle='round,pad=0.006,rounding_size=0.018',
        facecolor='white', edgecolor=BORDER, linewidth=1.0))
    ax.add_patch(FancyBboxPatch(
        (x, y0+h-0.105), w, 0.105,
        boxstyle='round,pad=0.006,rounding_size=0.018',
        facecolor=accent, edgecolor=accent, linewidth=0.8))
    ax.add_patch(Rectangle((x, y0+h-0.105), w, 0.026,
                           facecolor=accent, edgecolor='none'))
    ax.text(x+0.014, y0+h-0.052, f'{number:02d}', color='white', fontsize=5.8,
            fontweight='bold', ha='left', va='center')
    ax.text(x+0.055, y0+h-0.052, title, color='white', fontsize=5.4,
            fontweight='bold', ha='left', va='center')


def small_pill(x, y, w, text, edge, fill='#F8FAFC', fontsize=6.3):
    ax.add_patch(FancyBboxPatch(
        (x, y), w, 0.064, boxstyle='round,pad=0.003,rounding_size=0.016',
        facecolor=fill, edgecolor=edge, linewidth=0.75))
    ax.text(x+w/2, y+0.032, text, ha='center', va='center',
            fontsize=fontsize, color=NAVY, fontweight='bold')


titles = ['DATA', 'REPRESENTATION', 'LEAKAGE-FREE LOSO', 'DUAL EVALUATION']
for x, w, number, title, accent in zip(xs, ws, range(1, 5), titles, ACCENTS):
    stage_card(x, w, number, title, accent)

# Stage 1: cohort and temporal units.
x, w = xs[0], ws[0]
wave_x = np.linspace(x+0.025, x+w-0.025, 120)
wave_y = y0+0.585 + 0.025*np.sin(np.linspace(0, 7*np.pi, 120))
ax.plot(wave_x, wave_y, color=ACCENTS[0], lw=1.15)
ax.text(x+w/2, y0+0.505, 'CHB-MIT', ha='center', va='center',
        fontsize=6.4, fontweight='bold', color=NAVY)
ax.text(x+w/2, y0+0.425, f'{N_SUBJECTS} subjects  ·  {N_SEIZURES} seizures',
        ha='center', va='center', fontsize=4.7, color=MUTED)
small_pill(x+0.020, y0+0.285, w-0.040, '10-s epochs',
           ACCENTS[0], fontsize=4.8)
small_pill(x+0.020, y0+0.185, w-0.040, '30-min preictal',
           ACCENTS[0], fontsize=4.8)

# Stage 2: feature families.
x, w = xs[1], ws[1]
feature_rows = [
    ('F1', 'Handcrafted EEG', '#EAF3FC', '#2F80ED'),
    ('F2–F4', 'PLI / wPLI graphs', '#E8F6F3', '#1B998B'),
    ('F5–F7', 'Block-balanced fusion', '#F3EFF8', '#7B61A8'),
]
for idx, (code_label, label, fill, color) in enumerate(feature_rows):
    yy = y0+0.515-idx*0.125
    ax.add_patch(FancyBboxPatch(
        (x+0.020, yy), w-0.040, 0.087,
        boxstyle='round,pad=0.004,rounding_size=0.012',
        facecolor=fill, edgecolor='none'))
    ax.text(x+0.030, yy+0.044, code_label, ha='left', va='center',
            fontsize=5.2, fontweight='bold', color=color)
    ax.text(x+0.095, yy+0.044, label, ha='left', va='center',
            fontsize=4.3, color=NAVY)
ax.text(x+w/2, y0+0.145, 'Mutual information  ·  Top 20',
        ha='center', va='center', fontsize=4.8, color=MUTED, fontweight='bold')

# Stage 3: held-out ownership and training-only operations.
x, w = xs[2], ws[2]
ax.text(x+w/2, y0+0.570, 'Outer seizure-wise LOSO', ha='center', va='center',
        fontsize=5.4, fontweight='bold', color=NAVY)
fold_y = y0+0.480
for j in range(4):
    xx = x+0.033+j*0.043
    fill = ACCENTS[2] if j == 3 else '#F6F8FA'
    text_color = 'white' if j == 3 else MUTED
    ax.add_patch(FancyBboxPatch(
        (xx, fold_y), 0.032, 0.075,
        boxstyle='round,pad=0.002,rounding_size=0.008',
        facecolor=fill, edgecolor=ACCENTS[2], linewidth=0.7))
    ax.text(xx+0.016, fold_y+0.0375, str(j+1), ha='center', va='center',
            fontsize=4.8, color=text_color, fontweight='bold')
ax.text(x+w/2, y0+0.420, 'held-out seizure', ha='center', va='center',
        fontsize=4.6, color=MUTED)
ax.plot([x+0.025, x+w-0.025], [y0+0.365]*2, color=BORDER, lw=0.7)
ax.text(x+w/2, y0+0.310, 'Balance 1:1  ·  Scale  ·  MI', ha='center', va='center',
        fontsize=5.0, color=NAVY, fontweight='bold')
small_pill(x+0.055, y0+0.210, w-0.110, 'TRAINING FOLDS ONLY',
           ACCENTS[2], fill='#FFF5E8', fontsize=4.3)
ax.text(x+w/2, y0+0.145, 'Grouped inner CV for XGB / SVM', ha='center', va='center',
        fontsize=4.5, color=MUTED)

# Stage 4: models feed two evaluation targets.
x, w = xs[3], ws[3]
small_pill(x+0.028, y0+0.555, w-0.056, 'RF   ·   XGB   ·   SVM-RBF',
           ACCENTS[3], fill='#F4F0F8', fontsize=5.0)
eval_cards = [
    (y0+0.335, 'EPOCH-LEVEL', 'BAcc  ·  MCC  ·  F1',
     '180 preictal + 540 interictal', '#EDF4FB', '#2F80ED'),
    (y0+0.105, 'CONTINUOUS STREAM', 'SS  ·  FAR/h  ·  TTW',
     'causal alarm + refractory', '#EDF8F5', '#1B998B'),
]
for yy, heading, metrics, note, fill, color in eval_cards:
    ax.add_patch(FancyBboxPatch(
        (x+0.025, yy), w-0.050, 0.180,
        boxstyle='round,pad=0.005,rounding_size=0.014',
        facecolor=fill, edgecolor=color, linewidth=0.8))
    ax.text(x+0.040, yy+0.139, heading, ha='left', va='center',
            fontsize=5.0, color=color, fontweight='bold')
    ax.text(x+0.040, yy+0.088, metrics, ha='left', va='center',
            fontsize=5.4, color=NAVY, fontweight='bold')
    ax.text(x+0.040, yy+0.039, note, ha='left', va='center',
            fontsize=4.4, color=MUTED)

# One unambiguous left-to-right reading direction.
for i in range(3):
    arrow(ax, (xs[i]+ws[i]+0.004, 0.50),
          (xs[i+1]-0.004, 0.50), color=NAVY)

save_mbec_figure(fig, 'Fig1', 174, 92, 'Fig1')
plt.show()


In [ ]:
# ============================================================================
# 2. FIGURE 2 — TEMPORAL LABELING AND CAUSAL ALARM MECHANISM
# ============================================================================
fig, axes = plt.subplots(2, 1, gridspec_kw={'height_ratios': [0.9, 1.2]})

# (a) Retrospective labels and FAR eligibility.
ax = axes[0]
panel_label(ax, 'a')
ax.set_xlim(0, 100)
ax.set_ylim(-0.5, 2.1)
ax.axis('off')
segments = [
    (0, 24, 'Eligible\ninterictal', '#B8E0D2'),
    (24, 35, 'Excluded', '#D9D9D9'),
    (35, 65, 'Preictal\n30 min', '#F8C8C8'),
    (65, 73, 'Ictal', '#D84A4A'),
    (73, 86, 'Postictal', '#F2B6A0'),
    (86, 92, 'Gap', '#FFFFFF'),
    (92, 100, 'Eligible\ninterictal', '#B8E0D2'),
]
for start, end, label, color in segments:
    hatch = '//' if label == 'Gap' else None
    ax.add_patch(Rectangle((start, 0.55), end-start, 0.8,
                           facecolor=color, edgecolor='#455A64', hatch=hatch, lw=0.8))
    ax.text((start+end)/2, 0.95, label, ha='center', va='center',
            fontsize=7.3, color='white' if label == 'Ictal' else 'black',
            fontweight='bold' if label in {'Ictal', 'Preictal\n30 min'} else 'normal')
ax.annotate('', xy=(65, 1.62), xytext=(35, 1.62),
            arrowprops=dict(arrowstyle='<->', color='#C62828', lw=1.1))
ax.text(50, 1.78, 'SOP = 30 min; SPH = 0', ha='center', fontsize=8)
ax.plot([0, 24], [0.25, 0.25], color='#00897B', lw=3)
ax.plot([92, 100], [0.25, 0.25], color='#00897B', lw=3)
ax.text(12, 0.03, 'EligibleForFAR = 1', ha='center', fontsize=7.2, color='#00695C')
ax.text(96, 0.03, 'EligibleForFAR = 1', ha='center', fontsize=7.2, color='#00695C')

# (b) Five-minute firing power, true alarm, TTW and refractory.
ax = axes[1]
panel_label(ax, 'b')
ax.set_xlim(-8, 47)
ax.set_ylim(-1.15, 2.35)
ax.set_yticks([])
ax.set_xlabel('Monitoring time (min)')
ax.set_xticks([0, 10, 20, 30, 40])
ax.grid(False)
for spine in ['left', 'top', 'right']:
    ax.spines[spine].set_visible(False)

# Exactly 21 positive predictions among 30 epochs, shown schematically.
positive_idx = set([0,1,2,4,5,6,8,9,10,11,13,14,15,17,18,20,21,22,24,27,29])
for i in range(30):
    x = i * (5/30)
    color = '#D55E00' if i in positive_idx else '#DCEAF7'
    ax.add_patch(Rectangle((x, 0.55), 5/30-0.015, 0.55,
                           facecolor=color, edgecolor='white', lw=0.2))
ax.text(4.0, 1.40, '5-min causal window\n21/30 positive epochs',
        ha='right', va='center', fontsize=7.2, fontweight='bold',
        linespacing=1.15)
# Event lines are intentionally interrupted around their labels.  This avoids
# the common publication artefact where a vertical line cuts through text.
ax.plot([5, 5], [-1.15, 1.53], color='#1F4E79', lw=2, zorder=2)
ax.plot([5, 5], [1.92, 2.35], color='#1F4E79', lw=2, zorder=2)
ax.text(5, 1.72, 'ALARM', ha='center', va='center', color='#1F4E79',
        fontweight='bold', zorder=5)

# Refractory lockout and subsequent re-arming.
ax.add_patch(Rectangle((5, 0.55), 30, 0.55, facecolor='#DDF2D8',
                       edgecolor='#6ABF69', lw=0.9))
ax.text(14.7, 0.82, '30-min refractory lockout\nalarm history reset',
        ha='center', va='center', fontsize=6.9, linespacing=1.10,
        color='#2E7D32')
ax.add_patch(Rectangle((35, 0.55), 5, 0.55, facecolor='#FFF3CD',
                       edgecolor='#D4A017', lw=0.9))
ax.text(37.5, 0.82, 'Re-arm', ha='center', fontsize=7.2)

onset = 25
ax.plot([onset, onset], [-1.15, 1.53], color='#C62828', lw=2.4, zorder=2)
ax.plot([onset, onset], [1.92, 2.35], color='#C62828', lw=2.4, zorder=2)
ax.scatter([onset], [1.22], marker='^', s=80, color='#C62828', clip_on=False)
ax.text(onset, 1.72, 'SEIZURE ONSET', ha='center', va='center',
        color='#C62828', fontweight='bold', zorder=5)
ax.annotate('', xy=(onset, -0.2), xytext=(5, -0.2),
            arrowprops=dict(arrowstyle='<->', color='#6A3D9A', lw=1.3))
ax.text((onset+5)/2, -0.43, 'TTW = onset − first valid alarm',
        ha='center', fontsize=8, color='#5E2A84')
ax.text(4.0, -0.84, 'False alarm if trigger occurs\nin eligible interictal time',
        ha='right', va='center', fontsize=6.9, linespacing=1.12,
        color='#455A64')

save_mbec_figure(fig, 'Fig2', 174, 125, 'Fig2')
plt.show()


In [ ]:
# ============================================================================
# 3. FIGURE 3 — EPOCH-LEVEL PERFORMANCE HEATMAPS
# ============================================================================
fig, axes = plt.subplots(2, 1)
metrics = [('BalancedAccuracy', 'Balanced accuracy'), ('MCC', 'MCC')]
for panel, (ax, (metric, label)) in enumerate(zip(axes, metrics)):
    matrix = (pooled.pivot(index='Model', columns='FeatureSet', values=metric)
              .reindex(index=MODEL_ORDER, columns=FEATURE_ORDER))
    matrix.index = [MODEL_SHORT[x] for x in matrix.index]
    # Full feature-set names remain in the caption; compact codes are used in
    # the artwork to preserve legibility at the journal's final print width.
    matrix.columns = [x.split('_')[0] for x in matrix.columns]
    if metric == 'BalancedAccuracy':
        vmin, vmax, cmap = 0.45, max(0.75, float(matrix.max().max())), 'YlGnBu'
    else:
        max_abs = max(abs(float(matrix.min().min())), abs(float(matrix.max().max())), 0.4)
        vmin, vmax, cmap = -max_abs, max_abs, 'vlag'
    sns.heatmap(matrix, ax=ax, annot=True, fmt='.3f', cmap=cmap,
                vmin=vmin, vmax=vmax, linewidths=0.5, linecolor='white',
                annot_kws={'fontsize': 7.2},
                cbar_kws={'shrink': 0.82, 'pad': 0.018, 'label': label})
    ax.set_xlabel('Feature set (F1–F7)' if panel == 1 else '')
    ax.set_ylabel('Classifier')
    ax.tick_params(axis='x', rotation=0)
    ax.tick_params(axis='y', rotation=0)
    panel_label(ax, chr(ord('a') + panel))
fig.subplots_adjust(left=0.16, right=0.95, top=0.95, bottom=0.11, hspace=0.48)

save_mbec_figure(fig, 'Fig3', 174, 125, 'Fig3')
plt.show()


In [ ]:
# ============================================================================
# 4. FIGURE 4 — PRIMARY SEIZURE-SENSITIVITY–FAR TRADE-OFF
# ============================================================================
def pareto_front(frame):
    # Non-dominated when minimizing FAR and maximizing seizure sensitivity.
    rows = []
    for _, row in frame.iterrows():
        dominated = ((frame['FAR_per_h'] <= row['FAR_per_h']) &
                     (frame['SeizureSensitivity'] >= row['SeizureSensitivity']) &
                     ((frame['FAR_per_h'] < row['FAR_per_h']) |
                      (frame['SeizureSensitivity'] > row['SeizureSensitivity']))).any()
        if not dominated:
            rows.append(row)
    return pd.DataFrame(rows).sort_values('FAR_per_h')


label_offsets = {
    'F1_Handcrafted': (-18, -12), 'F2_Graph_PLI': (-5, 8),
    'F3_Graph_wPLI': (6, 6), 'F4_Graph_PLI_wPLI': (-18, -14),
    'F5_Handcrafted_PLI': (-19, 8), 'F6_Handcrafted_wPLI': (4, 10),
    'F7_All': (6, -11),
}

fig, axes = plt.subplots(1, 3, sharey=True)
for i, (ax, model_name) in enumerate(zip(axes, MODEL_ORDER)):
    data = (pooled.loc[pooled['Model'] == model_name]
            .set_index('FeatureSet').reindex(FEATURE_ORDER).reset_index())
    for _, row in data.iterrows():
        fs = row['FeatureSet']
        ax.scatter(row['FAR_per_h'], row['SeizureSensitivity'], s=58,
                   color=FEATURE_COLORS[fs], marker=FEATURE_MARKERS[fs],
                   edgecolor='black', linewidth=0.45, zorder=3)
        ax.annotate(fs.split('_')[0], (row['FAR_per_h'], row['SeizureSensitivity']),
                    xytext=label_offsets[fs], textcoords='offset points', fontsize=6.6,
                    bbox=dict(boxstyle='round,pad=0.12', fc='white', ec='none'))
    frontier = pareto_front(data)
    if len(frontier) > 1:
        ax.plot(frontier['FAR_per_h'], frontier['SeizureSensitivity'],
                color='#424242', lw=0.9, ls='--', zorder=2)
    ax.set_title(MODEL_SHORT[model_name], fontweight='bold')
    ax.set_xlabel('FAR (alarms/h)')
    xmax = float(data['FAR_per_h'].max())
    ax.set_xlim(-0.025, xmax * 1.18 + 0.02)
    ax.set_ylim(0.45, 0.74)
    ax.grid(True, axis='both')
    panel_label(ax, chr(ord('a') + i))
axes[0].set_ylabel('Seizure sensitivity')

legend_handles = [Line2D([0], [0], marker=FEATURE_MARKERS[f], color='none',
                         markerfacecolor=FEATURE_COLORS[f], markeredgecolor='black',
                         markersize=6, label=FEATURE_SHORT[f]) for f in FEATURE_ORDER]
fig.legend(handles=legend_handles, loc='lower center', ncol=4,
           bbox_to_anchor=(0.5, 0.025), frameon=False, columnspacing=1.4,
           handletextpad=0.5)
fig.subplots_adjust(left=0.09, right=0.985, top=0.90, bottom=0.34, wspace=0.26)

save_mbec_figure(fig, 'Fig4', 174, 105, 'Fig4')
plt.show()


In [ ]:
# ============================================================================
# 5. FIGURE 5 — WITHIN-PATIENT INCREMENTAL VALUE: F1 VERSUS F7 MCC
# ============================================================================
subject_order = sorted(subjects_df['Subject'].unique())
fig, axes = plt.subplots(1, 3, sharex=True, sharey=True)
for i, (ax, model_name) in enumerate(zip(axes, MODEL_ORDER)):
    data = subjects_df.loc[
        (subjects_df['Model'] == model_name) &
        (subjects_df['FeatureSet'].isin(['F1_Handcrafted', 'F7_All'])),
        ['Subject', 'FeatureSet', 'MCC']]
    wide = data.pivot(index='Subject', columns='FeatureSet', values='MCC').dropna()
    wide = wide.reindex(subject_order)
    delta = wide['F7_All'] - wide['F1_Handcrafted']
    y = np.arange(len(delta))
    colors = np.where(delta >= 0, '#2A7F62', '#C44E52')
    ax.hlines(y, 0, delta, color=colors, lw=1.2, zorder=1)
    ax.scatter(delta, y, color=colors, s=30, edgecolor='white', linewidth=0.5, zorder=2)
    ax.axvline(0, color='#424242', lw=0.9, ls='--')
    ax.axvline(float(delta.mean()), color='#5E3C99', lw=1.2, ls=':')
    ax.set_yticks(y)
    if i == 0:
        ax.set_yticklabels(subject_order)
    else:
        ax.tick_params(axis='y', labelleft=False)
    ax.invert_yaxis()
    ax.set_xlim(-0.56, 0.56)
    ax.set_xlabel('ΔMCC (F7 − F1)')
    ax.grid(True, axis='x')
    ax.set_title(MODEL_SHORT[model_name], fontweight='bold')
    panel_label(ax, chr(ord('a') + i))
axes[0].set_ylabel('Patient')
legend_handles = [
    Line2D([0], [0], marker='o', color='none', markerfacecolor='#2A7F62',
           markeredgecolor='white', label='F7 improved MCC'),
    Line2D([0], [0], marker='o', color='none', markerfacecolor='#C44E52',
           markeredgecolor='white', label='F7 reduced MCC'),
    Line2D([0], [0], color='#5E3C99', ls=':', label='Mean ΔMCC'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=3,
           bbox_to_anchor=(0.5, 0.02))
fig.subplots_adjust(left=0.12, right=0.985, top=0.90, bottom=0.22, wspace=0.18)

save_mbec_figure(fig, 'Fig5', 174, 105, 'Fig5')
plt.show()


In [ ]:
# ============================================================================
# 6. MANDATORY GRAPHICAL ABSTRACT — EXACT MBEC PRINT SIZE
# ============================================================================
fig, ax = plt.subplots(figsize=(GA_WIDTH, GA_HEIGHT))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

# A deliberately sparse, flat infographic: at the journal's mandatory small
# print size, hierarchy and whitespace are more useful than additional text.
navy, blue, teal, orange = '#173B57', '#2F80ED', '#168C80', '#F2994A'
ax.add_patch(Rectangle((0, 0), 1, 1, facecolor='#F7F9FC', edgecolor='none'))
ax.text(0.50, 0.952, 'EEG SEIZURE FORECASTING', ha='center', va='top',
        fontsize=5.2, fontweight='bold', color=navy)
ax.plot([0.34, 0.66], [0.900, 0.900], color=teal, lw=1.2,
        solid_capstyle='round')

# Two complementary feature views.
for x0, color in [(0.055, blue), (0.550, orange)]:
    ax.add_patch(FancyBboxPatch(
        (x0, 0.695), 0.395, 0.170,
        boxstyle='round,pad=0.010,rounding_size=0.025',
        facecolor='white', edgecolor=color, linewidth=0.65))

x = np.linspace(0.105, 0.410, 120)
y = (0.797 + 0.020*np.sin(np.linspace(0, 7*np.pi, x.size))
     + 0.007*np.sin(np.linspace(0, 18*np.pi, x.size)))
ax.plot(x, y, color=blue, lw=0.85, solid_capstyle='round')
ax.text(0.260, 0.731, 'LOCAL EEG', ha='center', va='center',
        fontsize=3.8, fontweight='bold', color=navy)

nodes = np.array([[0.625,0.817],[0.743,0.837],[0.865,0.813],
                  [0.820,0.770],[0.675,0.778]])
for a, b in [(0,1),(1,2),(2,3),(3,4),(4,0),(0,3),(1,3)]:
    ax.plot(nodes[[a,b],0], nodes[[a,b],1], color='#9CAAB4', lw=0.45,
            zorder=2)
ax.scatter(nodes[:,0], nodes[:,1], s=7.5, color=orange,
           edgecolor='white', linewidth=0.35, zorder=3)
ax.text(0.748, 0.715, 'BRAIN NETWORK', ha='center', va='bottom',
        fontsize=3.35, fontweight='bold', color=navy)
ax.text(0.500, 0.780, '+', ha='center', va='center', fontsize=5.0,
        fontweight='bold', color=navy)

# Leakage-controlled evaluation pipeline.
ax.add_patch(FancyBboxPatch(
    (0.055, 0.415), 0.890, 0.205,
    boxstyle='round,pad=0.010,rounding_size=0.025',
    facecolor=navy, edgecolor='none'))
ax.text(0.500, 0.560, 'SEIZURE-WISE LOSO', ha='center', va='center',
        fontsize=5.0, fontweight='bold', color='white')
ax.text(0.500, 0.503, 'leakage-free  ·  gap-aware', ha='center', va='center',
        fontsize=3.6, color='#DCE9F0')
ax.text(0.500, 0.448, 'RF   ·   XGB   ·   SVM-RBF', ha='center', va='center',
        fontsize=3.7, fontweight='bold', color='#8FE0D7')

# Separate epoch discrimination from clinically meaningful alarm behaviour.
for x0, color, title, metric in [
        (0.055, blue, 'EPOCH', 'MCC'),
        (0.550, teal, 'ALARM', 'SS · FAR/h · TTW')]:
    ax.add_patch(FancyBboxPatch(
        (x0, 0.145), 0.395, 0.170,
        boxstyle='round,pad=0.010,rounding_size=0.025',
        facecolor='white', edgecolor=color, linewidth=0.65))
    ax.text(x0 + 0.1975, 0.277, title, ha='center', va='center',
            fontsize=3.0, fontweight='bold', color=color)
    ax.plot([x0 + 0.145, x0 + 0.250], [0.258, 0.258],
            color=color, lw=0.7, solid_capstyle='round')
    ax.text(x0 + 0.1975, 0.205, metric, ha='center', va='center',
            fontsize=4.2 if title == 'EPOCH' else 3.15,
            fontweight='bold', color=navy)
ax.text(0.500, 0.226, '≠', ha='center', va='center', fontsize=7.0,
        fontweight='bold', color=orange)

ax.text(0.500, 0.070, 'DISCRIMINATION  ≠  CLINICAL UTILITY',
        ha='center', va='center', fontsize=3.45, fontweight='bold', color=navy)

save_mbec_figure(fig, 'Graphical_Abstract_Bui', 32.93, 37.63,
                 'GraphicalAbstract', graphical_abstract=True)
plt.show()


In [ ]:
# ============================================================================
# 7. SUPPLEMENTARY FIGURE S1 — ALARM-RULE SENSITIVITY FOR F1 AND F7
# ============================================================================
alarm_rule_order = [
    'Legacy_90_consecutive', 'FP_10min_theta_0.70', 'FP_5min_theta_0.80',
    'FP_5min_theta_0.70', 'FP_2min_theta_0.70', 'FP_5min_theta_0.50',
]
fig, axes = plt.subplots(1, 3, sharey=True)
for i, (ax, model_name) in enumerate(zip(axes, MODEL_ORDER)):
    for fs, ls in [('F1_Handcrafted', '-'), ('F7_All', '--')]:
        data = alarm_pooled.loc[
            (alarm_pooled['Model'] == model_name) &
            (alarm_pooled['FeatureSet'] == fs) &
            (alarm_pooled['AlarmRule'].isin(alarm_rule_order))].copy()
        data['AlarmRule'] = pd.Categorical(
            data['AlarmRule'], categories=alarm_rule_order, ordered=True)
        data = data.sort_values('AlarmRule')
        ax.plot(data['FAR_per_h'], data['SeizureSensitivity'], ls=ls,
                marker='o', ms=4, lw=1.1, color=FEATURE_COLORS[fs],
                label=FEATURE_SHORT[fs])
        for _, row in data.iterrows():
            if int(row['IsPrimary']) == 1:
                ax.scatter(row['FAR_per_h'], row['SeizureSensitivity'],
                           s=65, facecolors='none', edgecolors='black', lw=1.0, zorder=4)
    ax.set_title(MODEL_SHORT[model_name], fontweight='bold')
    ax.set_xlabel('FAR (alarms/h)')
    ax.set_ylim(-0.03, 1.03)
    ax.grid(True)
    panel_label(ax, chr(ord('a') + i))
axes[0].set_ylabel('Seizure sensitivity')
handles = [
    Line2D([0], [0], color=FEATURE_COLORS['F1_Handcrafted'], marker='o',
           label='F1 handcrafted'),
    Line2D([0], [0], color=FEATURE_COLORS['F7_All'], marker='o', ls='--',
           label='F7 all features'),
    Line2D([0], [0], marker='o', color='black', markerfacecolor='none',
           lw=0, label='Primary alarm rule'),
]
fig.legend(handles=handles, loc='lower center', ncol=3, bbox_to_anchor=(0.5, 0.02))
fig.subplots_adjust(left=0.09, right=0.985, top=0.90, bottom=0.23, wspace=0.28)
save_mbec_figure(fig, 'FigS1', 174, 95, 'FigS1')
plt.show()


In [ ]:
# ============================================================================
# 8. SUPPLEMENTARY FIGURE S2 — FEATURE-SELECTION STABILITY
# ============================================================================
def compact_feature_name(name):
    replacements = {
        'Handcrafted_': '', 'Graph_PLI_': 'PLI ', 'Graph_wPLI_': 'wPLI ',
        'mean_spectral_power': 'mean power', 'max_spectral_power': 'max power',
        'zero_crossing_rate': 'zero-crossing', '_Strength': ' strength',
    }
    clean = name
    for old, new in replacements.items():
        clean = clean.replace(old, new)
    clean = clean.replace('_', ' ')
    return clean if len(clean) <= 38 else clean[:35] + '…'


fig, axes = plt.subplots(2, 1)
for i, (ax, fs) in enumerate(zip(axes, ['F1_Handcrafted', 'F7_All'])):
    data = selected_features.loc[selected_features['FeatureSet'] == fs].copy()
    total_folds = data[['Subject', 'HeldOutSeizure']].drop_duplicates().shape[0]
    freq = (data.groupby(['Feature', 'FeatureBlock'])
            .size().rename('Count').reset_index())
    freq['SelectionFrequency_pct'] = 100 * freq['Count'] / total_folds
    freq = freq.sort_values(['SelectionFrequency_pct', 'Feature'],
                            ascending=[False, True]).head(12).sort_values('SelectionFrequency_pct')
    block_colors = {
        'Handcrafted': '#0072B2', 'Graph_PLI': '#E69F00', 'Graph_wPLI': '#009E73'}
    labels = [compact_feature_name(x) for x in freq['Feature']]
    ax.barh(labels, freq['SelectionFrequency_pct'],
            color=[block_colors.get(x, '#777777') for x in freq['FeatureBlock']])
    ax.set_xlim(0, 100)
    ax.set_xlabel('Selected outer folds (%)')
    ax.set_title(FEATURE_SHORT[fs], fontweight='bold')
    ax.tick_params(axis='y', labelsize=6.7)
    ax.grid(True, axis='x')
    panel_label(ax, chr(ord('a') + i))
legend = [Rectangle((0,0),1,1,facecolor=c,label=b) for b,c in block_colors.items()]
fig.legend(handles=legend, loc='lower center', ncol=3, frameon=False,
           bbox_to_anchor=(0.5, 0.015))
fig.subplots_adjust(left=0.31, right=0.97, top=0.95, bottom=0.20, hspace=0.54)
save_mbec_figure(fig, 'FigS2', 174, 160, 'FigS2')
plt.show()


In [ ]:
# ============================================================================
# 9. SUPPLEMENTARY FIGURE S3 — INNER-CV HYPERPARAMETER SELECTION
# ============================================================================
fig, axes = plt.subplots(1, 2)
for i, (ax, model_name) in enumerate(zip(axes, ['XGBoost', 'SVM_RBF'])):
    data = tuning.loc[tuning['Model'] == model_name].copy()
    # Stable short codes avoid unreadable JSON labels.
    unique_params = sorted(data['BestParams'].dropna().unique())
    code_map = {p: f'P{j+1}' for j, p in enumerate(unique_params)}
    data['ParamCode'] = data['BestParams'].map(code_map)
    table = pd.crosstab(data['FeatureSet'], data['ParamCode'], normalize='index') * 100
    table = table.reindex(FEATURE_ORDER).fillna(0)
    table.index = [x.split('_')[0] for x in table.index]
    sns.heatmap(table, ax=ax, annot=True, fmt='.0f', cmap='Blues', vmin=0, vmax=100,
                linewidths=0.4, cbar=i == 1, annot_kws={'fontsize': 7},
                cbar_kws={'label': 'Outer folds (%)'} if i == 1 else None)
    ax.set_title(MODEL_SHORT[model_name], fontweight='bold')
    ax.set_xlabel('Selected parameter configuration')
    ax.set_ylabel('Feature set' if i == 0 else '')
    panel_label(ax, chr(ord('a') + i))
    mapping_path = OUTPUT_ROOT / f'{model_name}_parameter_code_map.csv'
    pd.DataFrame({'ParamCode': list(code_map.values()),
                  'BestParams': list(code_map.keys())}).to_csv(mapping_path, index=False)
fig.subplots_adjust(left=0.10, right=0.90, top=0.90, bottom=0.18, wspace=0.42)
save_mbec_figure(fig, 'FigS3', 174, 100, 'FigS3')
plt.show()


In [ ]:
# ============================================================================
# 10. SUPPLEMENTARY FIGURE S4 — CLASSIFIER INFERENCE TIME
# ============================================================================
plot_timing = timing.copy()
fig, ax = plt.subplots()
y_base = np.arange(len(FEATURE_ORDER))
offsets = {'RandomForest': -0.20, 'XGBoost': 0.0, 'SVM_RBF': 0.20}
# Light connectors make the three classifier timings directly comparable
# within each feature representation.
timing_wide = (plot_timing.pivot(index='FeatureSet', columns='Model',
                                 values='MeanStreamInference_ms_per_epoch')
               .reindex(FEATURE_ORDER))
for y, (_, row) in enumerate(timing_wide.iterrows()):
    ax.hlines(y, float(row.min()), float(row.max()), color='#CBD4DC',
              lw=1.1, zorder=1)
for model_name in MODEL_ORDER:
    data = (plot_timing.loc[plot_timing['Model'] == model_name]
            .set_index('FeatureSet').reindex(FEATURE_ORDER))
    ax.scatter(data['MeanStreamInference_ms_per_epoch'], y_base + offsets[model_name],
               s=34, color=MODEL_COLORS[model_name], edgecolor='white', linewidth=0.5,
               label=MODEL_SHORT[model_name], zorder=3)
ax.set_xscale('log')
ax.set_yticks(y_base, [FEATURE_SHORT[x] for x in FEATURE_ORDER])
ax.invert_yaxis()
ax.set_xlabel('Classifier inference time (ms/epoch, log scale)')
ax.set_ylabel('Feature representation')
ax.grid(True, axis='x', which='both')
ax.legend(ncol=3, loc='lower center', bbox_to_anchor=(0.5, 1.01),
          borderaxespad=0, columnspacing=2.0, handletextpad=0.5)
fig.subplots_adjust(left=0.22, right=0.97, top=0.84, bottom=0.18)
save_mbec_figure(fig, 'FigS4', 174, 105, 'FigS4')
plt.show()


In [ ]:
# ============================================================================
# 11. CAPTIONS, MANIFEST AND FINAL QC
# ============================================================================
captions = {
    'Fig1': (
        'Fig. 1 Overall leakage-controlled framework for patient-specific seizure '
        'prediction. Ten-second EEG epochs were represented by handcrafted, PLI- '
        'and wPLI-derived graph, and block-balanced fused feature sets. Models were '
        'evaluated using outer leave-one-seizure-out validation, with preprocessing, '
        'balancing, feature selection, and grouped inner model selection restricted '
        'to the training data. Epoch-level classification and continuous-stream alarm '
        'performance were evaluated separately'),
    'Fig2': (
        'Fig. 2 Temporal labeling and causal alarm evaluation. (a) Retrospective EEG '
        'states determine the 30-min preictal target and the interictal monitoring time '
        'eligible for false-alarm-rate calculation. (b) The primary alarm is generated '
        'when at least 21 of 30 consecutive 10-s predictions are positive. The first '
        'valid alarm defines time to warning, and a 30-min refractory lockout follows '
        'each alarm'),
    'Fig3': (
        'Fig. 3 Pooled epoch-level performance across classifiers and feature '
        'representations. (a) Balanced accuracy. (b) Matthews correlation coefficient'),
    'Fig4': (
        'Fig. 4 Primary event-level seizure-sensitivity–false-alarm-rate trade-off '
        'for (a) random forest, (b) XGBoost, and (c) radial-basis-function support '
        'vector machine. Dashed segments connect non-dominated operating points'),
    'Fig5': (
        'Fig. 5 Within-patient change in Matthews correlation coefficient between '
        'handcrafted features alone (F1) and the block-balanced full fusion (F7) for '
        '(a) random forest, (b) XGBoost, and (c) radial-basis-function support vector '
        'machine'),
    'GraphicalAbstract': (
        'Graphical Abstract Local EEG and network-level representations are compared '
        'under seizure-wise, gap-aware evaluation to distinguish epoch-level '
        'discrimination from practical alarm utility'),
    'FigS1': (
        'Online Resource Fig. S1 Alarm-rule sensitivity for handcrafted (F1) and full '
        'fusion (F7) representations across classifiers. Open circles identify the '
        'primary 5-min firing-power threshold of 0.70'),
    'FigS2': (
        'Online Resource Fig. S2 Selection frequency of the 12 most recurrent '
        'features across outer folds for (a) F1 and (b) F7'),
    'FigS3': (
        'Online Resource Fig. S3 Frequency of grouped-inner-cross-validation '
        'hyperparameter selections for (a) XGBoost and (b) radial-basis-function '
        'support vector machine'),
    'FigS4': (
        'Online Resource Fig. S4 Mean classifier inference time per epoch using '
        'precomputed selected features. Feature extraction time is not included'),
}

with open(OUTPUT_ROOT / 'figure_captions.txt', 'w', encoding='utf-8') as file:
    for key, caption in captions.items():
        file.write(f'{key}\n{caption}\n\n')

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(OUTPUT_ROOT / 'figure_manifest.csv', index=False)

expected = {'Fig1', 'Fig2', 'Fig3', 'Fig4', 'Fig5',
            'Graphical_Abstract_Bui', 'FigS1', 'FigS2', 'FigS3', 'FigS4'}
observed = set(manifest['Figure'])
if observed != expected:
    raise AssertionError(f'Figure set mismatch: missing={expected-observed}, extra={observed-expected}')
for stem in expected:
    for ext in ['eps', 'tiff', 'png']:
        path = OUTPUT_ROOT / f'{stem}.{ext}'
        if not path.exists() or path.stat().st_size == 0:
            raise AssertionError(f'Missing or empty artwork: {path}')

print(manifest.to_string(index=False))
print('\nMBEC figure-generation QC: PASS')
print('Output directory:', OUTPUT_ROOT)
